In [ ]:
# 必要なパッケージのインストール
!pip install -q numpy pandas matplotlib tqdm

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm

# プロジェクトのルートをパスに追加
sys.path.insert(0, '.')

# データ拡張モジュールをインポート
from src.dataset.augmentation.pose_augmentation import (
    PoseAugmentation,
    AugmentationConfig,
    NormalizedPoseData
)

print("✓ モジュールのインポートが完了しました")

In [ ]:
# ===== ここを編集してください =====

# 入力CSVファイル（骨格データ）
INPUT_CSV = 'output/player_pose_data.csv'

# 出力ディレクトリ
OUTPUT_DIR = 'output/augmented_pose_data'

# データ拡張設定
AUGMENTATION_CONFIG = {
    # 左右反転
    'horizontal_flip': True,
    'horizontal_flip_prob': 0.5,
    
    # 回転
    'rotation': True,
    'rotation_range': 10.0,  # ±10度
    
    # スケーリング
    'scaling': True,
    'scale_range': (0.95, 1.05),  # ±5%
    
    # ガウシアンノイズ
    'add_noise': True,
    'noise_std': 0.015,
    
    # 関節ドロップアウト
    'keypoint_dropout': True,
    'dropout_prob': 0.05,  # 5%の確率で各関節がドロップアウト
    
    # 時間的拡張
    'temporal_scaling': True,
    'temporal_scale_range': (0.9, 1.1),  # ±10%の速度変化
    'temporal_jitter': True,
    'jitter_std': 0.5
}

# 各元データから生成する拡張データ数
NUM_AUGMENTATIONS_PER_SAMPLE = 5

# ランダムシード（再現性のため）
RANDOM_SEED = 42

# ===== ここまで =====

print("データ拡張パラメータ:")
print(f"  入力CSV: {INPUT_CSV}")
print(f"  出力ディレクトリ: {OUTPUT_DIR}")
print(f"  サンプルあたりの拡張数: {NUM_AUGMENTATIONS_PER_SAMPLE}")
print(f"  ランダムシード: {RANDOM_SEED}")
print(f"\n拡張設定:")
for key, value in AUGMENTATION_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# COCO形式のキーポイント名
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

def normalize_keypoints(keypoints: np.ndarray, confidences: np.ndarray) -> tuple:
    """
    キーポイントを正規化する（腰中心を原点、腰幅でスケール）
    
    Args:
        keypoints: キーポイント座標 (17, 2)
        confidences: 信頼度 (17,)
        
    Returns:
        (normalized_keypoints, hip_center, scale_factor)
    """
    # 左腰(11)と右腰(12)の座標
    left_hip = keypoints[11]
    right_hip = keypoints[12]
    left_hip_conf = confidences[11]
    right_hip_conf = confidences[12]
    
    # 両方の腰が検出されている場合
    if left_hip_conf > 0.3 and right_hip_conf > 0.3:
        # 腰の中心を計算
        hip_center = (left_hip + right_hip) / 2.0
        
        # 腰幅を計算（スケール係数）
        hip_width = np.linalg.norm(left_hip - right_hip)
        scale_factor = hip_width if hip_width > 0 else 1.0
        
        # 正規化: (座標 - 腰中心) / 腰幅
        normalized_keypoints = (keypoints - hip_center) / scale_factor
        
        return normalized_keypoints, tuple(hip_center), scale_factor
    else:
        # 腰が検出されていない場合はそのまま返す
        return keypoints.copy(), (0.0, 0.0), 1.0

def load_pose_data_from_csv(csv_path: str) -> List[NormalizedPoseData]:
    """
    CSVファイルから骨格データを読み込む
    
    Args:
        csv_path: CSVファイルのパス
        
    Returns:
        NormalizedPoseDataのリスト
    """
    df = pd.read_csv(csv_path)
    pose_data_list = []
    
    print(f"CSVファイルを読み込んでいます: {csv_path}")
    print(f"  総行数: {len(df)}")
    print(f"  列数: {len(df.columns)}")
    
    # CSVのフォーマットを確認
    has_normalized = 'nose_norm_x' in df.columns
    has_raw = 'nose_x' in df.columns
    
    if has_normalized:
        # 正規化済みデータの場合
        print("  フォーマット: 正規化済み骨格データ")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="データ読み込み中"):
            # キーポイント座標を抽出
            keypoints = np.zeros((17, 2), dtype=np.float32)
            confidences = np.zeros(17, dtype=np.float32)
            
            for i, name in enumerate(KEYPOINT_NAMES):
                keypoints[i, 0] = row[f'{name}_norm_x']
                keypoints[i, 1] = row[f'{name}_norm_y']
                confidences[i] = row[f'{name}_conf']
            
            # NormalizedPoseDataを作成
            pose_data = NormalizedPoseData(
                track_id=int(row['track_id']),
                frame=int(row['frame']),
                timestamp=float(row['timestamp']),
                normalized_keypoints=keypoints,
                keypoint_confidences=confidences,
                hip_center=(float(row['hip_center_x']), float(row['hip_center_y'])),
                scale_factor=float(row['scale_factor']),
                confidence=float(row['confidence']),
                is_valid=bool(row['is_valid'])
            )
            pose_data_list.append(pose_data)
    
    elif has_raw:
        # 生データの場合（正規化を適用）
        print("  フォーマット: 生の骨格データ（正規化を適用します）")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="データ読み込み中"):
            # キーポイント座標を抽出
            keypoints = np.zeros((17, 2), dtype=np.float32)
            confidences = np.zeros(17, dtype=np.float32)
            
            for i, name in enumerate(KEYPOINT_NAMES):
                keypoints[i, 0] = row[f'{name}_x']
                keypoints[i, 1] = row[f'{name}_y']
                confidences[i] = row[f'{name}_conf']
            
            # キーポイントを正規化
            normalized_keypoints, hip_center, scale_factor = normalize_keypoints(keypoints, confidences)
            
            # データの有効性をチェック（信頼度の高いキーポイントが十分あるか）
            valid_keypoints = np.sum(confidences > 0.3)
            is_valid = valid_keypoints >= 8  # 最低8個のキーポイントが必要
            
            # NormalizedPoseDataを作成
            pose_data = NormalizedPoseData(
                track_id=int(row['track_id']),
                frame=int(row['frame']),
                timestamp=float(row['timestamp']),
                normalized_keypoints=normalized_keypoints,
                keypoint_confidences=confidences,
                hip_center=hip_center,
                scale_factor=scale_factor,
                confidence=float(row['confidence']),
                is_valid=is_valid
            )
            pose_data_list.append(pose_data)
    
    else:
        # 対応していないフォーマット
        print("  エラー: 対応していないCSVフォーマットです")
        print("  必要なカラム: {keypoint_name}_x, {keypoint_name}_y, {keypoint_name}_conf")
        return []
    
    print(f"\n✓ {len(pose_data_list)} フレームのデータを読み込みました")
    
    # track_id別の統計
    track_ids = df['track_id'].unique()
    print(f"\ntrack_id別の統計:")
    for track_id in sorted(track_ids):
        count = len(df[df['track_id'] == track_id])
        print(f"  ID {track_id}: {count} フレーム")
    
    return pose_data_list

# データを読み込み
pose_data_list = load_pose_data_from_csv(INPUT_CSV)

In [ ]:
def augment_pose_dataset(
    pose_data_list: List[NormalizedPoseData],
    config_dict: Dict,
    num_augmentations: int,
    random_seed: int = None
) -> List[NormalizedPoseData]:
    """
    骨格データセットにデータ拡張を適用
    
    Args:
        pose_data_list: 元の骨格データリスト
        config_dict: 拡張設定の辞書
        num_augmentations: 各サンプルから生成する拡張数
        random_seed: ランダムシード
        
    Returns:
        拡張されたデータを含む骨格データリスト
    """
    augmented_data_list = []
    
    # 元データを保持
    augmented_data_list.extend(pose_data_list)
    
    print(f"\nデータ拡張を実行中...")
    print(f"  元データ数: {len(pose_data_list)}")
    print(f"  拡張数/サンプル: {num_augmentations}")
    print(f"  予想される総データ数: {len(pose_data_list) * (1 + num_augmentations)}\n")
    
    # 拡張設定を作成
    config = AugmentationConfig(**config_dict, random_seed=random_seed)
    augmentor = PoseAugmentation(config)
    
    # 各サンプルに対して拡張を適用
    for pose_data in tqdm(pose_data_list, desc="データ拡張中"):
        for aug_idx in range(num_augmentations):
            # 拡張を適用
            augmented_pose = augmentor.augment(pose_data)
            augmented_data_list.append(augmented_pose)
    
    print(f"\n✓ データ拡張完了")
    print(f"  総データ数: {len(augmented_data_list)} (元データ + 拡張データ)")
    
    return augmented_data_list

# データ拡張を実行
if len(pose_data_list) > 0:
    augmented_data_list = augment_pose_dataset(
        pose_data_list,
        AUGMENTATION_CONFIG,
        NUM_AUGMENTATIONS_PER_SAMPLE,
        RANDOM_SEED
    )
else:
    print("エラー: データが読み込まれていません")

In [ ]:
def visualize_pose_comparison(original: NormalizedPoseData, augmented: NormalizedPoseData):
    """
    元のポーズと拡張後のポーズを比較表示
    """
    # スケルトン接続定義
    skeleton_connections = [
        (0, 1), (0, 2), (1, 3), (2, 4),  # 顔
        (0, 5), (0, 6), (5, 6),           # 肩
        (5, 7), (7, 9),                   # 左腕
        (6, 8), (8, 10),                  # 右腕
        (5, 11), (6, 12), (11, 12),       # 体幹
        (11, 13), (13, 15),               # 左脚
        (12, 14), (14, 16)                # 右脚
    ]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    for ax, pose_data, title in zip(axes, [original, augmented], ['元データ', '拡張後']):
        # スケルトンを描画
        for connection in skeleton_connections:
            kp1_idx, kp2_idx = connection
            kp1 = pose_data.normalized_keypoints[kp1_idx]
            kp2 = pose_data.normalized_keypoints[kp2_idx]
            conf1 = pose_data.keypoint_confidences[kp1_idx]
            conf2 = pose_data.keypoint_confidences[kp2_idx]
            
            if conf1 > 0.3 and conf2 > 0.3:
                ax.plot([kp1[0], kp2[0]], [kp1[1], kp2[1]], 'b-', linewidth=2)
        
        # キーポイントを描画
        for i, kp in enumerate(pose_data.normalized_keypoints):
            conf = pose_data.keypoint_confidences[i]
            if conf > 0.3:
                ax.plot(kp[0], kp[1], 'ro', markersize=5)
        
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_aspect('equal')
        ax.invert_yaxis()  # Y軸を反転（画像座標系に合わせる）
        ax.grid(True, alpha=0.3)
        ax.set_title(title)
        ax.set_xlabel('X (正規化座標)')
        ax.set_ylabel('Y (正規化座標)')
    
    plt.tight_layout()
    plt.show()

# サンプルを可視化
if len(pose_data_list) > 0 and len(augmented_data_list) > len(pose_data_list):
    print("\n元データと拡張データの比較:")
    
    # ランダムに5つのサンプルを選択
    num_samples = min(5, len(pose_data_list))
    sample_indices = np.random.choice(len(pose_data_list), num_samples, replace=False)
    
    for idx in sample_indices:
        original = pose_data_list[idx]
        # 対応する拡張データ（最初の拡張）
        augmented = augmented_data_list[len(pose_data_list) + idx * NUM_AUGMENTATIONS_PER_SAMPLE]
        
        print(f"\nサンプル {idx + 1}: Frame {original.frame}, Track ID {original.track_id}")
        visualize_pose_comparison(original, augmented)

In [ ]:
if len(augmented_data_list) > 0:
    print("データ拡張の統計:")
    print(f"  元データ数: {len(pose_data_list)}")
    print(f"  拡張データ数: {len(augmented_data_list) - len(pose_data_list)}")
    print(f"  総データ数: {len(augmented_data_list)}")
    print(f"  拡張率: {len(augmented_data_list) / len(pose_data_list):.2f}x")
    
    # 有効性の統計
    valid_count = sum(1 for pose in augmented_data_list if pose.is_valid)
    print(f"\n有効なデータ:")
    print(f"  有効データ数: {valid_count}")
    print(f"  無効データ数: {len(augmented_data_list) - valid_count}")
    print(f"  有効率: {valid_count / len(augmented_data_list) * 100:.1f}%")
    
    # ドロップアウトの統計
    dropout_counts = []
    for pose in augmented_data_list:
        dropout_count = np.sum(pose.keypoint_confidences == 0)
        dropout_counts.append(dropout_count)
    
    print(f"\n関節ドロップアウトの統計:")
    print(f"  平均ドロップアウト数: {np.mean(dropout_counts):.2f} / 17関節")
    print(f"  最大ドロップアウト数: {np.max(dropout_counts)} / 17関節")
    print(f"  最小ドロップアウト数: {np.min(dropout_counts)} / 17関節")

## 8. 拡張データのエクスポート

In [ ]:
def export_augmented_data_to_csv(augmented_data_list: List[NormalizedPoseData], output_path: str):
    """
    拡張されたデータをCSVにエクスポート
    
    Args:
        augmented_data_list: 拡張されたデータのリスト
        output_path: 出力CSVパス
    """
    # DataFrameに変換
    data_dicts = [pose.to_dict() for pose in augmented_data_list]
    df = pd.DataFrame(data_dicts)
    
    # 出力ディレクトリを作成
    output_dir = Path(output_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # CSVに保存
    df.to_csv(output_path, index=False)
    
    print(f"\n✓ 拡張データをCSVに保存しました: {output_path}")
    print(f"  保存行数: {len(df)}")
    print(f"  ファイルサイズ: {Path(output_path).stat().st_size / 1024 / 1024:.2f} MB")

# エクスポート
if len(augmented_data_list) > 0:
    output_csv_path = f"{OUTPUT_DIR}/augmented_pose_data.csv"
    export_augmented_data_to_csv(augmented_data_list, output_csv_path)
    
    # 元データのみもエクスポート（比較用）
    original_csv_path = f"{OUTPUT_DIR}/original_pose_data.csv"
    export_augmented_data_to_csv(pose_data_list, original_csv_path)

In [ ]:
from google.colab import files

# 拡張データをダウンロード
output_csv_path = f"{OUTPUT_DIR}/augmented_pose_data.csv"
if Path(output_csv_path).exists():
    print("拡張データCSVをダウンロードしています...")
    files.download(output_csv_path)
    print("✓ ダウンロードが完了しました")
else:
    print("⚠ 拡張データCSVが見つかりません")